# 부족하면 웹에서 보강하는 RAG (Corrective RAG)

**Corrective RAG (CRAG)** 는 검색된 문서의 관련성을 평가해서, **부족하면 웹검색으로 보강** 한 뒤 답하는 패턴이다. 03 노트북이 "답변 후 평가" 중심이라면, 04 는 "답변 전에 문서를 거르고 모자라면 웹으로 채우는" 데 초점이 있다.

```
START → retrieve → grade_documents ──(충분)──▶ generate → END
                        │(부족, 일부 관련)──▶ web_search → generate
                        └(전부 무관)──▶ transform_query → web_search → generate
```

핵심: 검색 문서를 **하나씩 평가해 거르고**, 모자라면 질문을 웹검색용으로 다듬어 외부 정보를 덧붙인다.

> `OPENAI_API_KEY`, `TAVILY_API_KEY` 필요. 소스는 LangGraph 공식 문서(공개 URL).

## 환경 변수 준비

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
os.environ.setdefault("USER_AGENT", "ai-agent-study")
for k in ["OPENAI_API_KEY", "TAVILY_API_KEY"]:
    assert os.environ.get(k), f"{k} 가 .env 에 없습니다"
print("환경변수 로드 완료")

## 0. 여러 문서 로드 + 토큰 기반 청킹

여러 공개 문서(LangGraph 공식 문서 페이지들)를 한꺼번에 로드한다. [basics] `RecursiveCharacterTextSplitter.from_tiktoken_encoder` 는 글자 수가 아니라 **토큰 수** 기준으로 잘라, LLM 컨텍스트 한도에 맞추기 좋다.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader

urls = [
    "https://langchain-ai.github.io/langgraph/concepts/why-langgraph/",
    "https://langchain-ai.github.io/langgraph/agents/overview/",
]

loaded = [WebBaseLoader(url).load() for url in urls]
docs_list = [doc for sublist in loaded for doc in sublist]

# 토큰 250개 단위로 청킹
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=250, chunk_overlap=0
)
docs = text_splitter.split_documents(docs_list)
print(f"총 {len(docs)}개 청크")

## 1. 벡터스토어 + retriever
[basics] 임베딩 모델을 명시(`text-embedding-3-small`)할 수도 있다. k=3 으로 여러 문서를 가져온다.

In [ ]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

vectorstore = Chroma.from_documents(
    documents=docs, embedding=OpenAIEmbeddings(model="text-embedding-3-small")
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
print("retriever 준비 완료")

## Graph State
[basics] 이번엔 `MessagesState` 대신 순수 `TypedDict` State 를 쓴다 (메시지 누적이 핵심이 아니라 문서/질문 관리가 핵심).

In [ ]:
from typing import List
from typing_extensions import TypedDict

class State(TypedDict):
    question: str          # 질문 (재작성되면 갱신)
    generation: str        # 생성 답변
    document_lack: str     # 'Yes'/'No' — 문서가 부족한지
    documents: List[str]   # 관련 문서 목록

## Step 1. 노드

### 1) retrieve — 문서 검색

In [ ]:
def retrieve(state: State):
    print("##### RETRIEVE #####")
    question = state["question"]
    documents = retriever.invoke(question)
    return {"documents": documents, "question": question}

### 2) grade_documents — 문서를 하나씩 걸러내기

[basics] 구조화 출력으로 각 문서를 yes/no 평가한다. 관련 문서만 남기고, 하나라도 탈락하면 `document_lack='Yes'` 로 표시해 "웹 보강이 필요" 함을 알린다.

In [ ]:
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o", temperature=0)

class GradeDocuments(BaseModel):
    """문서 관련성 이진 점수"""
    binary_score: str = Field(description="질문에 관련된 문서면 'yes', 아니면 'no'")

grade_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You assess relevance of a retrieved document to a question. "
     "Grade 'yes' only if it clearly helps answer the question, else 'no'."),
    ("human", "Retrieved document:\n\n{document}\n\nQuestion: {question}"),
])
retrieval_grader = grade_prompt | llm.with_structured_output(GradeDocuments)

def grade_documents(state: State):
    print("##### GRADE DOCUMENTS #####")
    question, documents = state["question"], state["documents"]
    filtered_docs = []
    document_lack = "No"
    for d in documents:
        grade = retrieval_grader.invoke(
            {"question": question, "document": d.page_content}
        ).binary_score
        if grade == "yes":
            filtered_docs.append(d)
        else:
            document_lack = "Yes"   # 탈락한 문서가 있으면 웹 보강 필요
    print(f"관련 문서 {len(filtered_docs)}개, 보강 필요={document_lack}")
    return {"documents": filtered_docs, "question": question, "document_lack": document_lack}

### 3) transform_query — 웹검색용 질문 재작성

In [ ]:
def transform_query(state: State):
    print("##### TRANSFORM QUERY #####")
    question, documents = state["question"], state["documents"]
    rewrite_prompt = ChatPromptTemplate.from_messages([
        ("system", "You rewrite a question to be optimized for web search. "
                    "Reason about the underlying semantic intent."),
        ("human", "Initial question:\n\n{question}\nFormulate an improved question."),
    ])
    better = (rewrite_prompt | llm).invoke({"question": question})
    return {"documents": documents, "question": better.content}

### 4) web_search — 웹에서 정보 보강
[basics] Tavily 로 검색한 결과를 `Document` 로 만들어 기존 문서 목록에 덧붙인다.

In [ ]:
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.documents import Document

web_search_tool = TavilySearchResults(k=3)

def web_search(state: State):
    print("##### WEB SEARCH #####")
    question, documents = state["question"], state["documents"]
    results = web_search_tool.invoke({"query": question})
    web_content = "\n".join(d["content"] for d in results)
    documents.append(Document(page_content=web_content))   # 웹 결과를 문서로 추가
    return {"documents": documents, "question": question}

### 5) generate — 최종 답변
[basics] 모은 문서(내부 + 웹)를 근거로 답변 (프롬프트 인라인).

In [ ]:
RAG_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "You are an assistant for QA. Use the retrieved context to answer. "
     "If you don't know, say so. Three sentences max.\n\nContext:\n{context}"),
    ("human", "{question}"),
])

def generate(state: State):
    print("##### GENERATE #####")
    question, documents = state["question"], state["documents"]
    context = "\n\n".join(d.page_content for d in documents)
    response = llm.invoke(RAG_PROMPT.format_messages(context=context, question=question))
    return {"documents": documents, "question": question, "generation": response.content}

## Step 2. 엣지(라우터) — 문서 충분 여부로 분기

- 부족(`Yes`) + 관련 문서 0개 → `transform_query`(질문 재작성 후 웹)
- 부족(`Yes`) + 일부 관련 → `web_search_node`(웹으로 보강)
- 충분(`No`) → `generate`

In [ ]:
def decide_to_generate(state: State):
    print("##### ASSESS DOCUMENTS #####")
    if state["document_lack"] == "Yes":
        if len(state["documents"]) == 0:
            print("---ALL NOT RELEVANT → TRANSFORM QUERY---")
            return "transform_query"
        print("---SOME RELEVANT → WEB SEARCH---")
        return "web_search_node"
    print("---ENOUGH → GENERATE---")
    return "generate"

## Step 3. 그래프 컴파일

In [ ]:
from langgraph.graph import END, StateGraph, START

graph_builder = StateGraph(State)
graph_builder.add_node("retrieve", retrieve)
graph_builder.add_node("grade_documents", grade_documents)
graph_builder.add_node("generate", generate)
graph_builder.add_node("transform_query", transform_query)
graph_builder.add_node("web_search_node", web_search)

graph_builder.add_edge(START, "retrieve")
graph_builder.add_edge("retrieve", "grade_documents")
graph_builder.add_conditional_edges(
    "grade_documents", decide_to_generate,
    {"transform_query": "transform_query",
     "web_search_node": "web_search_node",
     "generate": "generate"},
)
graph_builder.add_edge("transform_query", "web_search_node")
graph_builder.add_edge("web_search_node", "generate")
graph_builder.add_edge("generate", END)
graph = graph_builder.compile()

In [ ]:
from IPython.display import Image, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    print(graph.get_graph().draw_mermaid())

## 테스트

### 내부 문서로 답 가능한 질문 (LangGraph 사용법)

In [ ]:
inputs = {"question": "What is LangGraph and why use it for agents?"}
for output in graph.stream(inputs):
    for key in output:
        print(f"Node '{key}' 완료")
print("\n[최종 답변]")
print(output["generate"]["generation"] if "generate" in output else output)

### 내부 문서에 없는 질문 → 웹 보강

In [ ]:
inputs = {"question": "Which companies use LangGraph in production?"}
for output in graph.stream(inputs):
    for key in output:
        print(f"Node '{key}' 완료")
last = output
print("\n[최종 답변]")
for v in last.values():
    if isinstance(v, dict) and v.get("generation"):
        print(v["generation"])

## 정리

- **Corrective RAG**: 검색 문서를 **하나씩 평가해 거르고**, 부족하면 **웹검색으로 보강**
- 분기: 전부 무관→질문 재작성→웹 / 일부 관련→웹 보강 / 충분→바로 생성
- 웹 결과도 `Document` 로 감싸 기존 문서와 함께 컨텍스트로 사용
- 03(답변 후 환각/해결성 평가) vs 04(답변 전 문서 보강) — 보완 단계가 다름

다음: 문서 검색이 아니라 **DB 에 SQL 을 작성·실행·수정** 하는 RAG.